In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score


# -------------------------------------------------
# Load and Merge 3 Heart Datasets
# -------------------------------------------------

cols = [
    'age',
    'sex',
    'cp',
    'trestbps',
    'chol',
    'fbs',
    'restecg',
    'thalach',
    'exang',
    'oldpeak',
    'slope',
    'ca',
    'thal',
    'target'
]

# Load datasets
df1 = pd.read_csv("Cleavland.csv", names=cols)
df2 = pd.read_csv("hung.csv", names=cols)
df3 = pd.read_csv("Switzerland.csv", names=cols)

# Merge datasets
df = pd.concat([df1, df2, df3], ignore_index=True)

print(df.head())


# -------------------------------------------------
# a. Data Cleaning
# -------------------------------------------------

# Replace ? with NaN
df.replace('?', np.nan, inplace=True)

# Convert columns to numeric
df = df.apply(pd.to_numeric, errors='coerce')

# Remove missing values
df.dropna(inplace=True)

# Remove negative values
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    df = df[df[col] >= 0]

print("\nCleaned Data:")
print(df.head())


# -------------------------------------------------
# b. Error Correcting (Outlier Detection)
# -------------------------------------------------

# Outlier removal using IQR method

feature_cols = df.columns.drop('target')

for col in feature_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df = df[(df[col] >= lower) & (df[col] <= upper)]

print("\nData after Outlier Removal:")
print(df.head())


# -------------------------------------------------
# c. Data Transformation
# -------------------------------------------------

# Features and Target
X = df.drop('target', axis=1)
y = df['target']

# Convert target into binary classes
# 0 = No disease
# 1 = Disease

y = np.where(y > 0, 1, 0)

# Standardization
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)


# -------------------------------------------------
# d. Build Models
# -------------------------------------------------

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

# Logistic Regression
lr_model = LogisticRegression()

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

lr_accuracy = accuracy_score(y_test, lr_pred)

print("\nLogistic Regression Accuracy:", lr_accuracy)


# kNN Model
knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model.fit(X_train, y_train)

knn_pred = knn_model.predict(X_test)

knn_accuracy = accuracy_score(y_test, knn_pred)

print("\nkNN Accuracy:", knn_accuracy)


# Compare Accuracy
if lr_accuracy > knn_accuracy:
    print("\nLogistic Regression performs better.")
else:
    print("\nkNN performs better.")

    age  sex   cp trestbps   chol  fbs restecg thalach exang oldpeak slope  \
0  63.0  1.0  1.0    145.0  233.0  1.0     2.0   150.0   0.0     2.3   3.0   
1  67.0  1.0  4.0    160.0  286.0  0.0     2.0   108.0   1.0     1.5   2.0   
2  67.0  1.0  4.0    120.0  229.0  0.0     2.0   129.0   1.0     2.6   2.0   
3  37.0  1.0  3.0    130.0  250.0  0.0     0.0   187.0   0.0     3.5   3.0   
4  41.0  0.0  2.0    130.0  204.0  0.0     2.0   172.0   0.0     1.4   1.0   

    ca thal  target  
0  0.0  6.0       0  
1  3.0  3.0       2  
2  2.0  7.0       1  
3  0.0  3.0       0  
4  0.0  3.0       0  

Cleaned Data:
    age  sex   cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0  63.0  1.0  1.0     145.0  233.0  1.0      2.0    150.0    0.0      2.3   
1  67.0  1.0  4.0     160.0  286.0  0.0      2.0    108.0    1.0      1.5   
2  67.0  1.0  4.0     120.0  229.0  0.0      2.0    129.0    1.0      2.6   
3  37.0  1.0  3.0     130.0  250.0  0.0      0.0    187.0    0.0      3.5   